<a href="https://colab.research.google.com/github/jialraro/AnalisisDatos/blob/main/TelecomxP2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import requests

url = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

# Realizar la solicitud a la API
response = requests.get(url)

# Validar que la respuesta sea exitosa
if response.status_code == 200:
    data = response.json()
    print("Datos cargados correctamente")
else:
    print(f" Error al cargar datos. Código: {response.status_code}")


Datos cargados correctamente


In [3]:
data = response.json()

df = pd.json_normalize(data)

In [4]:
print('Churn' in df.columns)

True


In [5]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

In [6]:
# 1. Obtener todas las columnas categóricas
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# 2. Eliminar 'Churn' si estuviera presente (aunque no lo esté, esto es seguro)
if 'Churn' in categorical_cols:
    categorical_cols.remove('Churn')

# 3. Aplicar One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


In [7]:
# Verificar proporción de cancelación
proporcion_churn = df_encoded['Churn'].value_counts(normalize=True) * 100
print("Proporción de cancelación (Churn):")
print(proporcion_churn)


Proporción de cancelación (Churn):
Churn
0.0    73.463013
1.0    26.536987
Name: proportion, dtype: float64


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Matriz de correlación
correlation_matrix = df_encoded.corr()

# Visualización
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, annot=False)
plt.title("Matriz de correlación entre variables")
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='account.Contract', hue='Churn')
plt.title('Relación entre Tipo de Contrato y Cancelación')
plt.xlabel('Tipo de Contrato')
plt.ylabel('Cantidad de Clientes')
plt.legend(title='Churn', labels=['No', 'Sí'])
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x='Churn', y='account.Charges.Total')
plt.title('Distribución del Gasto Total según Cancelación')
plt.xlabel('Cancelación')
plt.ylabel('Gasto Total')
plt.xticks([0, 1], ['No', 'Sí'])
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(data=df, x='account.Charges.Monthly', y='account.Charges.Total', hue='Churn', alpha=0.6)
plt.title('Relación Gasto Mensual vs Total según Cancelación')
plt.xlabel('Gasto Mensual')
plt.ylabel('Gasto Total')
plt.legend(title='Churn', labels=['No', 'Sí'])
plt.tight_layout()
plt.show()


In [ ]:
# Separar variables predictoras y variable objetivo
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']


In [ ]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto en entrenamiento (70%) y prueba (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Entrenar modelo
modelo_rf = RandomForestClassifier(random_state=42, class_weight='balanced')
modelo_rf.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Predicción
y_pred = modelo_rf.predict(X_test)

# Accuracy
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

# Reporte de clasificación
print("Reporte de Clasificación:")
print(classification_report(y_test, y_pred, target_names=['No Canceló', 'Canceló']))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No', 'Sí'], yticklabels=['No', 'Sí'])
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión - Random Forest')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Suponiendo que y_test = reales, y_pred = predichos por el modelo

def evaluar_modelo(y_test, y_pred, modelo_nombre="Modelo"):
    print(f"\n📊 Resultados para: {modelo_nombre}")
    print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precisión: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-score: {f1_score(y_test, y_pred):.4f}")

    print("\nMatriz de Confusión:")
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No", "Sí"], yticklabels=["No", "Sí"])
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.title(f"Matriz de Confusión – {modelo_nombre}")
    plt.tight_layout()
    plt.show()

    print("\nReporte de Clasificación:")
    print(classification_report(y_test, y_pred, target_names=["No Canceló", "Canceló"]))


In [ ]:
importances = modelo_rf.feature_importances_
importancia_variables = pd.Series(importances, index=X.columns).sort_values(ascending=False)

# Visualizar
importancia_variables.head(10).plot(kind='barh')
plt.title("Variables más importantes según Random Forest")
plt.gca().invert_yaxis()
plt.show()
